# Buck Converter Worksheet
#### Author: Daniel Whent
#### Version: 03/26/2026
#### Description: 

In [ ]:
import numpy as np

import plotly.graph_objects as go
import eseries as es

from pint import UnitRegistry
from sigfig import round as sigfig_round

import pandas as pd
ureg = UnitRegistry()

import warnings
warnings.filterwarnings("ignore")

def sf_round(value, sig_figs=3):
    return sigfig_round(value.magnitude, sig_figs) * value.units

In [ ]:
# set the setup variables
V_in = 33.9 * ureg.volt  # Input voltage in volts

V_out = 12 * ureg.volt  # Output voltage in volts

I_out = 1 * ureg.ampere  # Max output current in amperes

dV_out = 0.05 * ureg.volt  # Output voltage ripple in volts

f_sw = 1e6 * ureg.hertz  # Switching frequency in Hz

nu = 0.9  # Efficiency of the buck converter (estimated)

I_lim_pk = 2 * ureg.ampere  # current limit of the buck converter chosen (use the min value)

rr = 0.3  # Ripple ratio (ΔI_L / I_out) 

v_fb = 1.25 * ureg.volt

fb_i_bias = 0.1e-6 * ureg.ampere

Ripple ratio should be chosen based on the specific requirements of the application but will generally fall between 0.2 and 0.6
- for slower switchers the impact of a lower ripple ratio is more significant on the load transient response
- for switchers anticipating input voltage variations, a higher ripple ratio can help maintain a more stable output voltage
- for switchers with fast transient response requirements, a higher ripple ratio can help minimize output voltage deviations during load changes
- for switchers with high efficiency requirements, a lower ripple ratio can help reduce conduction losses and improve overall efficiency
THe effects of transients on the system are 

In [ ]:
D = V_out / (V_in * nu) # estimation of duty cycle max
L_target = ((V_out*I_out*(V_in-V_out))/(V_in*f_sw*rr*I_out)).magnitude*ureg.henry

print(f'Target inductance for Ripple Ratio ({rr}): {L_target.to_compact():~P}')

Select an inductor that close to the target value ensure that inductance is sufficient at max load current

In [ ]:
L_selected = 10e-6 * ureg.henry

dIl = ((V_out * (V_in - V_out)) / (L_selected * f_sw * V_in)).magnitude*ureg.ampere
Il_rms = dIl/(2*np.sqrt(3))
I_trip = I_lim_pk-dIl/2

print(f"Inductor current ripple for selected inductor ({L_selected.to_compact():~P}): {dIl.to_compact():~P}")
print(f'Inductor ripple current RMS = {Il_rms.to_compact():~P}')
print(f'Overcurrent trip point = {I_trip.to_compact():~P}')

In [ ]:
r_bot_max = (v_fb / (fb_i_bias * 100))

print(f"Maximum bottom resistor value for feedback network: {r_bot_max.to(ureg.kiloohm):~P}")

In [ ]:
# Find the 10 closest E-series resistor divider combos for V_out = v_fb * (1 + R_top/R_bot)
# Use E9 series for practical values

def find_resistor_dividers(v_out, v_fb, r_bot_max, series=es.E96, n=5):
    # Get E-series values in ohms
    e_vals = es.erange(series, 1e3, r_bot_max.to('ohm').magnitude+1)
    combos = []
    for r_bot in e_vals:
        if r_bot > r_bot_max.to('ohm').magnitude:
            continue
        r_top = (v_out/v_fb - 1) * r_bot
        # Find closest E-series value for r_top
        r_top_e = es.find_nearest(series, r_top)
        v_out_actual = v_fb * (1 + r_top_e/r_bot)
        error = (v_out_actual - v_out)/v_out
        current = sf_round(v_fb/r_bot,2)
        combos.append((r_top_e, r_bot, v_out_actual, error, current))
    # sort by current
    combos.sort(key=lambda x: x[4])
    del combos[len(combos)//10:]
    # Sort by error
    combos.sort(key=lambda x: abs(x[3]))
    return combos[:n]

combos = find_resistor_dividers(V_out.to('volt').magnitude, v_fb.to('volt').magnitude, r_bot_max)
df_combos = pd.DataFrame(combos, columns=["R_top (Ω)", "R_bot (Ω)", "V_out_actual (V)", "Error", "Current (µA)"])
df_combos["Error (%)"] = df_combos["Error"] * 100
df_combos = df_combos.drop(columns="Error")
df_combos["Current (µA)"] = df_combos["Current (µA)"] * 1e6
print("Top 10 closest E96 resistor divider combos:")
display(df_combos)

Some regulators have requirements for feed forward caps when using higher feedback resistor values Make sure to follow these guidelines

In [ ]:
selection = 0

rtop,rbot,vout_t = df_combos.iloc[selection][["R_top (Ω)", "R_bot (Ω)", "V_out_actual (V)"]]

print(f"Selected resistor divider: \n\tR_top = {rtop} Ω, \n\tR_bot = {rbot} Ω, \n\tV_out_actual = {sf_round(vout_t,5)} V")

Regulators with internal compensation should use input and output capacitors within the recomended range should be used, if using ceramics ensure that they have minimal ESR at the switching frequency, if using electrolytic or poly unsure that there are small ceramics close to the part to minimize the current loop size especially on the input side of the regulator.

Also, mostly for ceramic caps care should be taken to ensure that effective capacitance is sufficient with the expected dc bias

For electrolytic caps ensure that the rms current does not age the cap too fast, this can be mitigated by placing ceramics close to the part as the switching capcitors, and then increase distance between the regulator and the electrolytic capacitor to add stray inductance. this will take most of the "Switching load" off of the electrolytics. Also this is generally better for EMI.

In [ ]:
# assume that the component of the deltaVout equal between the ESR and the capacitor ripple

dv_ESR = dV_out/2
dv_C = dV_out/2

esr_max = dv_ESR/dIl

Cout_min = dIl/(8*f_sw*dv_C)

print(f"Maximum ESR: {esr_max.to(ureg.milliohm)}")
print(f"Minimum Effective Capacitance at {V_out.to_compact(ureg.volt)}: {Cout_min.to_compact(ureg.farad)}") 

## ESR decreases in parallel

Cout increases in parallel

In [ ]:
Cout_selected = 40e-6 * ureg.farad
esr_selected = 1.1e-3 * ureg.ohm

dv_esr_t = dIl * esr_selected
dv_c_t = dIl / (8 * f_sw * Cout_selected)


print(f"Voltage ripple due to ESR: {dv_esr_t.to(ureg.millivolt)}")
print(f"Voltage ripple due to capacitor: {dv_c_t.to(ureg.millivolt)}")
dv_total = dv_esr_t + dv_c_t
print(f"Total voltage ripple: {dv_total.to(ureg.millivolt):~P}")

In [ ]:
print(f"Vout...............{V_out.to(ureg.volt):~P}\n"+
      f"Iout_max...........{I_trip.to(ureg.ampere):~P}\n"+
      f"I_Lim..............{I_lim_pk.to(ureg.ampere):~P} Pk (min)\n"+
      f"ΔI.................{dIl.to(ureg.milliamp):~P}\n"+
      f"ΔVout..............{dv_total.to(ureg.millivolt):~P}\n"+
      f"fsw................{f_sw.to(ureg.megahertz):~P}")